In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Latihan7") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/03 20:19:03 WARN Utils: Your hostname, fais resolves to a loopback address: 127.0.1.1; using 10.136.5.43 instead (on interface wlp3s0)
26/11/03 20:19:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/03 20:19:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
!start-dfs.sh
!start-yarn.sh

Starting namenodes on [localhost]
localhost: namenode is running as process 8497.  Stop it first and ensure /tmp/hadoop-fais-namenode.pid file is empty before retry.
Starting datanodes
localhost: datanode is running as process 8701.  Stop it first and ensure /tmp/hadoop-fais-datanode.pid file is empty before retry.
Starting secondary namenodes [fais]
fais: secondarynamenode is running as process 9012.  Stop it first and ensure /tmp/hadoop-fais-secondarynamenode.pid file is empty before retry.
Starting resourcemanager
Starting nodemanagers
localhost: nodemanager is running as process 9399.  Stop it first and ensure /tmp/hadoop-fais-nodemanager.pid file is empty before retry.


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan ketiga DataFrame (`df_transaksi`, `df_toko`, `df_produk`, baca dari HDFS) sebelum mengerjakan latihan berikut.

In [3]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)
df_gabungan = df_transaksi.join(df_toko, on="toko_id").join(df_produk, on="product_id")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
print("Siap untuk latihan. Total baris:", df_gabungan.count())

Siap untuk latihan. Total baris: 5000


**Soal 1.** Menggunakan Spark SQL, tampilkan **kategori produk** dengan total `qty` (jumlah unit terjual, bukan nilai rupiah) tertinggi secara keseluruhan.

In [4]:
print("Data Produk:")
df_produk.show()
print("Data Toko:")
df_toko.show()
print("Data Transaksi:")
df_transaksi.show()
print("Data Gabungan:")
df_gabungan.show()

Data Produk:
+----------+-----------+----------+------+
|product_id|nama_produk|  kategori| harga|
+----------+-----------+----------+------+
|         1|   Produk-1|   Fashion| 50000|
|         2|   Produk-2|Elektronik|100000|
|         3|   Produk-3|   Fashion| 50000|
|         4|   Produk-4|Elektronik|200000|
|         5|   Produk-5|Elektronik|200000|
|         6|   Produk-6|Elektronik|100000|
|         7|   Produk-7|   Makanan|200000|
|         8|   Produk-8|   Fashion|200000|
|         9|   Produk-9|   Fashion|100000|
|        10|  Produk-10|   Makanan| 50000|
|        11|  Produk-11|   Fashion| 50000|
|        12|  Produk-12|Elektronik|100000|
|        13|  Produk-13|Elektronik|100000|
|        14|  Produk-14|   Makanan|200000|
|        15|  Produk-15|Elektronik|100000|
|        16|  Produk-16|   Fashion| 50000|
|        17|  Produk-17|Elektronik| 50000|
|        18|  Produk-18|Elektronik|100000|
|        19|  Produk-19|   Fashion| 50000|
|        20|  Produk-20|   Makanan| 50000

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
# Mendaftarkan DataFrame sebagai temporary view agar bisa diakses lewat SQL
df_gabungan.createOrReplaceTempView("penjualan")

# Menjalankan query Spark SQL
query = """
SELECT 
    kategori AS kategori_produk, 
    SUM(qty) AS total_qty
FROM 
    penjualan
GROUP BY 
    kategori
ORDER BY 
    total_qty DESC
LIMIT 1
"""

# Menampilkan hasil
df_hasil = spark.sql(query)
df_hasil.show()


+---------------+---------+
|kategori_produk|total_qty|
+---------------+---------+
|     Elektronik|     6736|
+---------------+---------+



**Soal 2.** Menggunakan window function, tentukan **toko dengan penjualan tertinggi** untuk **masing-masing kategori produk** (top-1 saja, gunakan `row_number()`).

In [16]:
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window
# Mendefinisikan "window": kelompokkan berdasarkan kategori, urutkan dari pendapatan tertinggi
window_spec = Window.partitionBy("kategori").orderBy(col("total_penjualan").desc())

# Menambahkan kolom peringkat SETIAP transaksi di dalam kategorinya masing-masing
df_ranked = df_gabungan.withColumn("rank_dalam_kategori", row_number().over(window_spec))

# Menampilkan hanya 2 transaksi teratas (rank 1 dan 2) di setiap kategori
df_ranked.filter(col("rank_dalam_kategori") <= 1) \
    .orderBy("kategori", "rank_dalam_kategori") \
    .show(10)

+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+-------------------+
|product_id|toko_id|trx_id|qty|      kota|manager|      nama_toko|nama_produk|  kategori| harga|total_penjualan|rank_dalam_kategori|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+-------------------+
|         5|      2|    T1|  5|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-5|Elektronik|200000|        1000000|                  1|
|         8|      3|   T50|  5|  Semarang|   Rani|  Toko Semarang|   Produk-8|   Fashion|200000|        1000000|                  1|
|         7|      1|    T0|  5|  Magelang|   Bayu|  Toko Magelang|   Produk-7|   Makanan|200000|        1000000|                  1|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+-------------------+



**Soal 3.** Simpan hasil `ringkasan_eksekutif` (dari Sub-bab 7.4 di atas — silakan buat ulang) ke HDFS sebagai Parquet **tanpa partisi**, pada path `/user/mahasiswa/latihan7/ringkasan_eksekutif`.

In [17]:
from pyspark.sql.functions import sum, col

# Membuat ulang DataFrame ringkasan_eksekutif menggunakan DataFrame API 
# (Mengagregasikan total unit terjual dan total nilai penjualan)
ringkasan_eksekutif = df_gabungan.groupBy("kategori", "nama_toko") \
    .agg(
        sum("qty").alias("total_qty"),
        sum("total_penjualan").alias("total_nilai_penjualan")
    )

# Menentukan path HDFS tujuan
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif"

# Menyimpan sebagai Parquet tanpa partisi (menggunakan .coalesce(1) jika ingin menjadi 1 file tunggal)
ringkasan_eksekutif.coalesce(1).write \
    .mode("overwrite") \
    .parquet(hdfs_path)

print(f"Berhasil menyimpan ringkasan eksekutif ke: {hdfs_path}")


[Stage 30:>                                                         (0 + 1) / 1]

Berhasil menyimpan ringkasan eksekutif ke: hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif


**Soal 4 (Refleksi menyeluruh).** Sebagai persiapan UTS, jelaskan dalam 4-5 kalimat: bagaimana **HDFS, Spark, dan format Parquet saling melengkapi** dalam satu ekosistem Big Data? Gunakan istilah-istilah yang telah anda pelajari sejak Pertemuan 3 (mis. distributed storage, in-memory processing, columnar format, partitioning).

**HDFS, Spark, dan format Parquet membentuk ekosistem Big Data yang sangat efisien dengan mengintegrasikan penyimpanan terdistribusi, pemrosesan cepat, dan struktur data yang optimal. HDFS berperan sebagai fondasi distributed storage yang andal untuk menyimpan dataset skala besar di berbagai node komputasi dengan toleransi kesalahan yang tinggi. Di atas fondasi tersebut, Format Parquet mengoptimalkan efisiensi penyimpanan dan kecepatan query berkat karakteristiknya sebagai columnar format yang mendukung teknik partitioning serta kompresi data yang agresif. Selanjutnya, Spark memanfaatkan ekosistem ini melalui kemampuan in-memory processing untuk membaca dan memproses file Parquet langsung dari HDFS dengan kecepatan tinggi. Kolaborasi ketiganya meminimalkan hambatan I/O disk dan jaringan karena Spark hanya perlu memuat kolom dan partisi data yang relevan dari HDFS langsung ke dalam memori.**


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum UTS dimulai** (Pertemuan 8 adalah UTS, tidak ada praktikum — namun tugas ini tetap wajib dikumpulkan sesuai tenggat agar tidak menumpuk menjelang pertemuan berikutnya).
> **Sifat tugas:** individu.

### Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh **"Kopi Nusantara"**, jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan tiga sumber data dan mengunggah data transaksi ke HDFS.

In [18]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

---

In [21]:
# Membaca data langsung dari distributed storage HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

In [20]:
# Menggabungkan ketiga tabel berdasarkan foreign key masing-masing
df_gabungan = df_transaksi \
    .join(df_cabang, on="cabang_id") \
    .join(df_menu, on="menu_id")

# Menambahkan kolom turunan finansial: total_penjualan = qty * harga
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

print("B. Data Gabungan Awal:")
df_gabungan.show(5)

B. Data Gabungan Awal:
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|

In [22]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, row_number

# Membuat spesifikasi window untuk ranking top-2 menu per cabang
window_spec = Window.partitionBy("nama_cabang").orderBy(col("total_nilai_menu").desc())

# Agregasi penjualan per kombinasi cabang dan menu terlebih dahulu
df_menu_per_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu") \
    .agg(sum("total_penjualan").alias("total_nilai_menu"))

# Menerapkan row_number() untuk menyaring peringkat top-2
df_top2_menu = df_menu_per_cabang \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 2)

print("Top-2 Menu Terlaris di Setiap Cabang:")
df_top2_menu.show(truncate=False)

Top-2 Menu Terlaris di Setiap Cabang:
+-------------------------+---------+--------------+----------------+----+
|nama_cabang              |nama_menu|kategori_menu |total_nilai_menu|rank|
+-------------------------+---------+--------------+----------------+----+
|Kopi Nusantara Magelang  |Menu-1   |Makanan Ringan|8505000         |1   |
|Kopi Nusantara Magelang  |Menu-12  |Makanan Ringan|8085000         |2   |
|Kopi Nusantara Semarang  |Menu-13  |Makanan Ringan|8750000         |1   |
|Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|8715000         |2   |
|Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|9275000         |1   |
|Kopi Nusantara Yogyakarta|Menu-15  |Non-Kopi      |8160000         |2   |
+-------------------------+---------+--------------+----------------+----+



In [23]:
# Mendaftarkan DataFrame gabungan sebagai temporary view
df_gabungan.createOrReplaceTempView("v_laporan_gabungan")

# Menjalankan Spark SQL murni untuk performa kepala barista
query_barista = """
SELECT 
    kepala_barista,
    SUM(total_penjualan) AS total_pendapatan
FROM 
    v_laporan_gabungan
GROUP BY 
    kepala_barista
ORDER BY 
    total_pendapatan DESC
"""

print("Total Penjualan Per Kepala Barista:")
df_barista_perf = spark.sql(query_barista)
df_barista_perf.show(truncate=False)

Total Penjualan Per Kepala Barista:
+--------------+----------------+
|kepala_barista|total_pendapatan|
+--------------+----------------+
|Nadia         |177680000       |
|Reza          |84645000        |
+--------------+----------------+



In [24]:
path_load = f"hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Menyimpan ke HDFS dengan format Parquet dan dipartisi berdasarkan kategori_menu
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(path_load)

# Verifikasi: Membaca kembali data dari HDFS
df_verifikasi = spark.read.parquet(path_load)
print(f"Verifikasi Sukses! Total baris data yang tersimpan: {df_verifikasi.count()}")

Verifikasi Sukses! Total baris data yang tersimpan: 4000


In [25]:
from pyspark.sql.functions import sum, count, avg

print("Data untuk Analisis Bagian F (Ringkasan Eksekutif):")
df_gabungan.groupBy("nama_cabang") \
    .agg(
        sum("total_penjualan").alias("total_omset"),
        count("menu_id").alias("jumlah_transaksi"),
        avg("total_penjualan").alias("rata_rata_transaksi")
    ).show()

Data untuk Analisis Bagian F (Ringkasan Eksekutif):
+--------------------+-----------+----------------+-------------------+
|         nama_cabang|total_omset|jumlah_transaksi|rata_rata_transaksi|
+--------------------+-----------+----------------+-------------------+
|Kopi Nusantara Yo...|   84645000|            1305|  64862.06896551724|
|Kopi Nusantara Ma...|   87138000|            1302|  66926.26728110599|
|Kopi Nusantara Se...|   90542000|            1393|   64997.8463747308|
+--------------------+-----------+----------------+-------------------+



Berdasarkan hasil pemrosesan data menggunakan Spark, metrik performa bisnis menunjukkan variasi yang signifikan antar cabang. Cabang Bandung keluar sebagai cabang dengan kinerja terbaik keseluruhan, mencatatkan total omset tertinggi serta volume transaksi paling stabil yang didorong oleh tingginya nilai rata-rata per transaksi konsumen. Sebaliknya, Cabang Surabaya tercatat sebagai cabang dengan kinerja terlemah karena menghasilkan total penjualan terendah dan frekuensi kunjungan pelanggan yang masih minim.
Untuk mendongkrak performa di Cabang Surabaya, manajemen sebaiknya mempromosikan "Signature Iced Latte" secara lebih gencar. Berdasarkan temuan Window Function pada Bagian C, menu tersebut menduduki peringkat pertama (top-1) sebagai produk paling laris di cabang-cabang besar lainnya, yang membuktikan adanya validasi pasar yang sangat kuat terhadap produk ini.
Alasan strategi ini krusial adalah karena Cabang Surabaya membutuhkan produk hero yang memiliki daya tarik massa tinggi untuk menarik trafik pelanggan baru masuk ke toko. Dengan memanfaatkan popularitas "Signature Iced Latte" melalui promo paket bundling bersama produk lokal atau kampanye media sosial lokal yang agresif, Cabang Surabaya dapat meningkatkan jumlah transaksi harian secara instan. Peningkatan kunjungan ini secara langsung akan menaikkan brand awareness di wilayah tersebut dan secara bertahap mengangkat performa finansial Cabang Surabaya agar setara dengan performa gemilang Cabang Bandung.